# TalentCLEF 2025 - Task B: Baseline for Test set

In this notebook, we provide the Test Set baseline for TalentCLEF Task B. This document includes downloading the [Task B dataset](https://doi.org/10.5281/zenodo.14002665), applying an embedding model as a baseline to generate the submission .trec file, which will then be compressed and uploaded to the Codalab platform.


-----------------------------
TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

This shared-task's inaugural edition is part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2025.clef-initiative.eu/index.php?page=Pages/labs.html), scheduled to be held in Madrid in 2025. If you are interested in registering, you can find registration form [here](https://clef2025-labs-registration.dei.unipd.it/).

<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2025/workshop/logo_clef_madrid.png" alt="TalentCLEF logo" width="150"/>


## Imports

In [2]:
!pip install codecarbon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 576.4/576.4 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 kB 8.1 MB/s eta 0:00:00
  Attempting uninstall: termcolor
    Found existing installation: termcolor 3.1.0
    Uninstalling termcolor-3.1.0:
      Successfully uninstalled termcolor-3.1.0
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take int

In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import subprocess
from codecarbon import EmissionsTracker

## Download Task B files

First, let's download the Task A and Task B zip files directly from Zenodo.



In [2]:
# Download
!wget https://zenodo.org/records/15292308/files/TaskB.zip
!unzip TaskB.zip -d taskB

--2025-05-08 20:55:33--  https://zenodo.org/records/15292308/files/TaskB.zip
Resolving zenodo.org (zenodo.org)... 188.185.48.194, 188.185.45.92, 188.185.43.25, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4556562 (4.3M) [application/octet-stream]
Saving to: ‘TaskB.zip’

TaskB.zip           100%[===================>]   4.34M  3.40MB/s    in 1.3s    

2025-05-08 20:55:35 (3.40 MB/s) - ‘TaskB.zip’ saved [4556562/4556562]

Archive:  TaskB.zip
   creating: taskB/test/
  inflating: taskB/test/queries      
  inflating: taskB/test/corpus_elements  
   creating: taskB/training/
  inflating: taskB/training/job2skill.tsv  
  inflating: taskB/training/jobid2terms.json  
  inflating: taskB/training/skillid2terms.json  
   creating: taskB/validation/
  inflating: taskB/validation/corpus_elements  
  inflating: taskB/validation/qrels.tsv  
  inflating: taskB/validation/queries  


- Data path

In [16]:
import os
import json

In [4]:
root_path="taskB"

In [5]:
training_data_path = os.path.join(root_path, "training")
validation_data_path=os.path.join(root_path, "validation")
test_data_path=os.path.join(root_path, "test")

## Processing training data

In [14]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(training_data_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])
job2skill.head()

,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


In [17]:
# Read json files
jobid2terms_path = os.path.join(training_data_path, "jobid2terms.json")
with open(jobid2terms_path, 'r') as file:
    jobid2terms = json.load(file)
#print (jobid2terms['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34'])

In [18]:
# Read json files
skillid2terms_path = os.path.join(training_data_path, "skillid2terms.json")
with open(skillid2terms_path, 'r') as file:
    skillid2terms = json.load(file)
#print (skillid2terms)

## Generate submission file using the baseline model

Load queries and corpus elements in English from the Validation folder:

In [19]:
queries = "/content/taskB/test/queries"
corpus_elements = "/content/taskB/test/corpus_elements"

In [20]:
queries = pd.read_csv(queries,sep="\t")
corpus_elements = pd.read_csv(corpus_elements, sep="\t")


Transform `skill_aliases` column to a list of strings:

In [21]:
import ast
corpus_elements["skill_aliases"] = corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

- Approaches to expand the document (skill esco_uri)

- Approach A

In [22]:
def related_skill_terms(terms_list):
    return " ".join(terms_list)

corpus_elements["doc_A"] = corpus_elements["skill_aliases"].apply(lambda x: related_skill_terms(x))

- Approach B

In [23]:
def get_essential_jobids(skill_id):
    return list(job2skill[(job2skill["skill_id"]==skill_id) & (job2skill["rel_type"]=="essential")]["job_id"])

def related_jobid_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)
    terms_B = []
    for related_jobid in related_essential_jobids:
        related_jobid_terms = jobid2terms[related_jobid]
        terms_B.extend(related_jobid_terms)
    return " ".join(terms_B)

corpus_elements["doc_B"] = corpus_elements["esco_uri"].apply(lambda x: related_jobid_terms(x))

- Approach C

In [24]:
def get_essential_skillids(job_id):
    return list(job2skill[(job2skill["job_id"]==job_id) & (job2skill["rel_type"]=="essential")]["skill_id"])

def related_job2skill_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)

    related_skillids = []
    for related_jobid in related_essential_jobids:
        skill_ids = get_essential_skillids(related_jobid)
        skill_ids.remove(skill_id)
        related_skillids.extend(skill_ids)

    terms_C = []
    for related_skillid in related_skillids:
        related_skillid_terms = skillid2terms[related_skillid]
        terms_C.extend(related_skillid_terms)

    return " ".join(terms_C)

corpus_elements["doc_C"] = corpus_elements["esco_uri"].apply(lambda x: related_job2skill_terms(x))

In [25]:
print (len(corpus_elements))
corpus_elements.columns

1986


Index(['c_id', 'esco_uri', 'skill_aliases', 'doc_A', 'doc_B', 'doc_C'], dtype='object')

Generate a mapping dictionary between IDs and texts from query and corpus element strings.

In [26]:
queries_ids = queries.q_id.to_list()
queries_texts = queries.jobtitle.to_list()
map_queries = dict(zip(queries_ids,queries_texts))

Before creating a mapping dictionary of texts to corpus_ids, explode the `skill_aliases` column:

In [27]:
len(queries), len(queries_ids), len(corpus_elements)

(1520, 1520, 1986)

In [28]:
list_aliases_df = corpus_elements.explode("skill_aliases")

In [29]:
corpus_ids = list_aliases_df.c_id.to_list()
#corpus_texts = list_aliases_df.skill_aliases.to_list()
#map_corpus = dict(zip(corpus_texts,corpus_ids))

Load simple embedding model:

In [30]:
model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [31]:
tracker = EmissionsTracker()
tracker.start_task("all-MiniLM-L6-v2")

Encode queries and corpus elements:

In [32]:
#doc_A
corpus_doc_A_texts = corpus_elements.doc_A.to_list()
#map_corpus_doc_A = dict(zip(corpus_ids, corpus_doc_A_texts))
#print (len(corpus_doc_A_texts), len(corpus_ids), len(map_corpus_doc_A))

#doc_B
corpus_doc_B_texts = corpus_elements.doc_B.to_list()
#map_corpus_doc_B = dict(zip(corpus_ids, corpus_doc_B_texts))
#print (len(corpus_doc_B_texts), len(corpus_ids), len(map_corpus_doc_B))

#doc_C
corpus_doc_C_texts = corpus_elements.doc_C.to_list()
#map_corpus_doc_C = dict(zip(corpus_ids, corpus_doc_C_texts))
#print (len(corpus_doc_C_texts), len(corpus_ids), len(map_corpus_doc_C))

In [33]:
query_embeddings = model.encode(queries_texts, convert_to_tensor=True)
#corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True)

In [34]:
corpus_doc_A_embedding = model.encode(corpus_doc_A_texts, convert_to_tensor=True)
corpus_doc_B_embedding = model.encode(corpus_doc_B_texts, convert_to_tensor=True)
corpus_doc_C_embedding = model.encode(corpus_doc_C_texts, convert_to_tensor=True)

Compute similarities

In [35]:
similarities_query_doc_A = util.cos_sim(query_embeddings, corpus_doc_A_embedding).cpu().numpy()

similarities_query_doc_B = util.cos_sim(query_embeddings, corpus_doc_B_embedding).cpu().numpy()

similarities_query_doc_C = util.cos_sim(query_embeddings, corpus_doc_C_embedding).cpu().numpy()

In [36]:
similarities_query_doc_A.shape, similarities_query_doc_B.shape, similarities_query_doc_C.shape

((1520, 1986), (1520, 1986), (1520, 1986))

In [37]:
#similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()
emissions = tracker.stop_task("all-MiniLM-L6-v2")

In [38]:
#similarities_query_doc_AB = similarities_query_doc_A + similarities_query_doc_B
similarities_query_doc_AC = similarities_query_doc_A + similarities_query_doc_C
#similarities_query_doc_BC = similarities_query_doc_B + similarities_query_doc_C
#similarities_query_doc_ABC = similarities_query_doc_A + similarities_query_doc_B + similarities_query_doc_C
similarities_query_doc_WABC = similarities_query_doc_A*0.5 + similarities_query_doc_B*0.3 + similarities_query_doc_C*0.2

## Prepare submission file

The submissions must follow the TREC Run File format, including headers in the output file. This means that the fle have 6 space-spearated columns per line, with following information:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved document.
- rank: Position of the document in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

In [49]:
#similarities = similarities_query_doc_A

#similarities = similarities_query_doc_B
#similarities = similarities_query_doc_C

similarities = similarities_query_doc_AC

#similarities = similarities_query_doc_WABC
model_name = "AC"

In [50]:
corpus_texts = corpus_doc_A_texts

In [51]:
import numpy as np
results = []
results_name = []

for q_idx, q_id in enumerate(queries_ids):
    sorted_indices = np.argsort(-similarities[q_idx])
    used_doc_ids = set()
    rank_counter = 0
    for c_idx in sorted_indices:  # Consider the full list.
        doc_id = corpus_ids[c_idx]
        # If doc_id was already processed, go to the next one.
        if doc_id in used_doc_ids:
            continue
        used_doc_ids.add(doc_id)
        rank_counter += 1

        query_name = map_queries[q_id]
        doc_name = corpus_texts[c_idx]
        score = similarities[q_idx, c_idx]

        results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
        results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} {model_name}")

Let's save the list as a file:

In [52]:
with open("run_testset_"+model_name+"_taskB.trec", "w", encoding="utf-8") as f:
    f.write("\n".join(results))

In [53]:
import json
json.dump(dict(emissions.values), open("./emissions.json", "w"), ensure_ascii=False, indent=4)

Zip the results that will be uploaded into the [Task B Codabench](https://www.codabench.org/competitions/7059)

In [54]:
!zip taskB_testset_ac.zip run_*
!zip taskB_testset_baseline_emissions.zip emissions.json

  adding: run_testset_AC_taskB.trec (deflated 78%)
updating: emissions.json (deflated 48%)
